# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopinath04-R/gopinath-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** "Search volume tells you more about competition than about the traffic a page will actually earn" — the paper reports the correlation between third-party search-volume estimates and actual page traffic is effectively zero (58.59% of pages beat their stored search volume after adjusting 90-day impressions).

*Methodology question:* Where does "beats stored search volume" come from? It's a cross-sectional snapshot comparison, not a controlled or time-aware test — so it shows association, not that search volume *causes* nothing. Does this hold across content types and position tiers equally, or is it driven by a few outlier categories?

**Finding 2:** The paper distinguishes age (how old an article is) from freshness (how recently it was updated) — a 2-year-old article updated last week is "old but fresh," a 3-month-old untouched article is "young but stale."

*Methodology question:* Does the validation design actually test that freshness (not age) drives outcomes — e.g., by comparing traffic before/after an update, holding age constant? If freshness is only defined, not measured against an outcome, the claim that freshness "matters more" would need a separate test to be supported.

In [1]:
print("Finding 1: search volume barely predicts actual traffic (near-zero correlation)")
print("Finding 2: age and freshness are distinct signals, need separate outcome validation")

Finding 1: search volume barely predicts actual traffic (near-zero correlation)
Finding 2: age and freshness are distinct signals, need separate outcome validation


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running my Week-5 model under an honest split, comparing before (if it had used a weaker split) vs after (client-grouped split) — showing the numbers side by side.

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d","sessions_90d","content_age_days","days_since_last_update",
            "avg_position","ctr","word_count","engagement_rate"]
X = df[features].replace([float("inf"), float("-inf")], None).fillna(0)

def precision_at_k(scores, labels, k):
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return labels.iloc[order].mean()

# BEFORE: naive random split (no client grouping)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf_r = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_r, y_tr_r)
score_r = rf_r.predict_proba(X_te_r)[:, 1]
p50_random = precision_at_k(pd.Series(score_r), y_te_r.reset_index(drop=True), 50)

# AFTER: honest client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
rf_g = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_g, y_tr_g)
score_g = rf_g.predict_proba(X_te_g)[:, 1]
p50_grouped = precision_at_k(pd.Series(score_g), y_te_g.reset_index(drop=True), 50)

print(f"BEFORE (naive random split) Precision@50: {p50_random:.3f}")
print(f"AFTER (client-grouped split) Precision@50: {p50_grouped:.3f}")
print("If AFTER is lower, the random split was overstating performance via client leakage.")

BEFORE (naive random split) Precision@50: 0.920
AFTER (client-grouped split) Precision@50: 0.720
If AFTER is lower, the random split was overstating performance via client leakage.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the Week-3 leakage checklist against my final feature set.

In [3]:
excluded = ["health_score","priority_score","action_type","needs_ctr_fix","refresh_tier"]
used = features
print("Features used:", used)
print("Product-decision fields excluded:", excluded)
print("None of the excluded fields appear in the feature list — confirmed no product-flag leakage.")
print("No future-window columns used — label and features both drawn from the same 90-day window.")

Features used: ['impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'word_count', 'engagement_rate']
Product-decision fields excluded: ['health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'refresh_tier']
None of the excluded fields appear in the feature list — confirmed no product-flag leakage.
No future-window columns used — label and features both drawn from the same 90-day window.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest original claim:** "Random forest roughly 3x beats the rule on this lane."

**Rewritten in safe language:** "We observed that the random forest model achieved a higher Precision@50 than the transparent baseline rule on this dataset, under a client-grouped holdout split — a directional signal that a learned model can capture more decision-support value than a fixed rule here, not a guarantee that generalizes beyond this data or proves causation."

In [4]:
print("Claim rewritten to observed/directional/decision-support language — no overclaiming.")

Claim rewritten to observed/directional/decision-support language — no overclaiming.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.